# Camada Gold — Organização dos Dados para Análise

Nesta etapa, os dados tratados na camada Silver serão organizados para responder às perguntas sobre o perfil dos clientes da Terra Signal.

O objetivo é preparar informações sobre contratos, serviços contratados, tempo de permanência e cancelamentos (churn), permitindo a construção de indicadores e análises exploratórias.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
arquivo_silver = Path("dados/silver/history_silver.csv")

df_gold = pd.read_csv(arquivo_silver)

df_gold.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,CustomerFeedback,MonthlyIncome,TotalCharges_Estimado,tenure_numerico
0,4578-PHJYZ,Male,0,Yes,Yes,52,Yes,No,DSL,No,...,One year,Yes,Electronic check,68.75,3482.85,No,I have been a customer with this internet prov...,6532,3482.85,52.0
1,6289-CPNLD,Male,0,Yes,Yes,33,Yes,No,DSL,No,...,One year,Yes,Mailed check,73.90,2405.05,Yes,I've been a customer with this company for ove...,7634,2405.05,33.0
2,2682-KEVRP,Female,1,No,No,22,Yes,No,No,No internet service,...,One year,Yes,Mailed check,20.05,417.00,No,I have been a customer with this company for 2...,3628,417.00,22.0
3,5697-GOMBF,Female,1,Yes,Yes,28,No,No phone service,DSL,No,...,Month-to-month,Yes,Electronic check,35.90,973.65,No,I have been a customer with this internet prov...,7851,973.65,28.0
4,9717-QEBGU,Male,0,No,No,2,Yes,No,Fiber optic,No,...,Month-to-month,Yes,Electronic check,70.30,144.00,No,I have been using the fiber optic internet ser...,1691,144.00,2.0


- arquivo_silver: define o caminho do arquivo.
- pd.read_csv(): carrega os dados da Silver.
- df_gold: será nosso DataFrame de trabalho na camada Gold.
- .head(): mostra os cinco primeiros registros para conferência.

Importante: estamos apenas carregando os dados, sem modificar o arquivo original.

### Análise dos cancelamentos de clientes

Será analisada a distribuição dos clientes que cancelaram ou permaneceram com os serviços da Terra Signal, utilizando a variável `Churn`.

In [3]:
df_gold["Churn"].value_counts()

Churn
No     4130
Yes    1504
Name: count, dtype: int64

In [4]:
df_gold["Churn"].value_counts(normalize=True) * 100

Churn
No     73.304934
Yes    26.695066
Name: proportion, dtype: float64

### Resultado da análise de cancelamentos

Dos 5.634 clientes analisados, 1.504 cancelaram os serviços (26,70%), enquanto 4.130 permaneceram (73,30%).

O resultado indica que aproximadamente um em cada quatro clientes da base apresenta cancelamento, tornando o churn um indicador importante para investigar possíveis relações com contratos, serviços e tempo de permanência.

### Análise dos cancelamentos por tipo de contrato

Será analisada a relação entre o tipo de contrato e o cancelamento dos serviços, buscando identificar quais modalidades apresentam maiores taxas de churn.

In [9]:


tabela_contratos = pd.crosstab(
    df_gold["Contract"],
    df_gold["Churn"]
)

tabela_contratos = tabela_contratos.rename(
    columns={"No": "Permaneceram", "Yes": "Cancelaram"}
)

tabela_contratos["Total"] = tabela_contratos.sum(axis=1)

tabela_contratos.index.name = "Tipo de contrato"
tabela_contratos.columns.name = None

tabela_contratos["Taxa de cancelamento (%)"] = (
    tabela_contratos["Cancelaram"] / tabela_contratos["Total"] * 100
).round(2)

tabela_contratos

,Permaneceram,Cancelaram,Total,Taxa de cancelamento (%)
Tipo de contrato,,,,
Month-to-month,1777,1322,3099,42.66
One year,1050,138,1188,11.62
Two year,1303,44,1347,3.27


### Resultado da análise de cancelamentos por tipo de contrato

Os contratos mensais apresentaram a maior taxa de cancelamento (42,66%), seguidos pelos contratos anuais (11,62%) e pelos contratos de dois anos (3,27%).

Os resultados indicam que clientes com contratos de maior duração apresentam menores taxas de cancelamento, sugerindo uma relação entre o tipo de contrato e a permanência dos clientes.

### Análise dos cancelamentos por serviço de internet

Será analisada a relação entre o serviço de internet contratado e a taxa de cancelamento dos clientes, buscando identificar quais modalidades apresentam maior proporção de churn.

In [11]:
tabela_internet = pd.crosstab(
    df_gold["InternetService"],
    df_gold["Churn"]
)

tabela_internet = tabela_internet.rename(
    columns={"No": "Permaneceram", "Yes": "Cancelaram"}
)

tabela_internet["Total"] = tabela_internet.sum(axis=1)

tabela_internet.index.name = "Serviço de internet"
tabela_internet.columns.name = None

tabela_internet["Taxa de cancelamento (%)"] = (
    tabela_internet["Cancelaram"] / tabela_internet["Total"] * 100
).round(2)

tabela_internet

,Permaneceram,Cancelaram,Total,Taxa de cancelamento (%)
Serviço de internet,,,,
DSL,1551,373,1924,19.39
Fiber optic,1435,1042,2477,42.07
No,1144,89,1233,7.22


### Resultado da análise de cancelamentos por serviço de internet

Os clientes com fibra óptica apresentaram a maior taxa de cancelamento (42,07%), seguidos pelos clientes DSL (19,39%) e pelos clientes sem serviço de internet (7,22%).

Os resultados indicam uma associação entre o tipo de serviço contratado e o cancelamento, com destaque para a maior proporção de churn entre os clientes de fibra óptica.

### Análise dos cancelamentos por tempo de permanência

Será analisada a relação entre o tempo de permanência dos clientes e os cancelamentos, comparando a média de meses dos clientes que permaneceram e daqueles que cancelaram os serviços.

In [12]:
df_gold.groupby("Churn")["tenure_numerico"].mean().round(2)

Churn
No     37.38
Yes    18.33
Name: tenure_numerico, dtype: float64

### Resultado da análise de cancelamentos por tempo de permanência

Os clientes que permaneceram apresentaram tempo médio de 37,38 meses, enquanto aqueles que cancelaram registraram média de 18,33 meses.

A diferença de 19,05 meses indica uma associação entre menor tempo de permanência e cancelamento dos serviços.

### Análise dos cancelamentos por faixa de permanência

Os clientes serão agrupados por faixas de tempo de permanência para comparar as taxas de cancelamento e identificar possíveis diferenças entre clientes mais recentes e antigos.

In [13]:
df_gold["Faixa_permanencia"] = pd.cut(
    df_gold["tenure_numerico"],
    bins=[-1, 12, 24, 48, 72],
    labels=["Até 12 meses", "13 a 24 meses", "25 a 48 meses", "Acima de 48 meses"]
)

df_gold["Faixa_permanencia"].value_counts().sort_index()

Faixa_permanencia
Até 12 meses         1751
13 a 24 meses         805
25 a 48 meses        1289
Acima de 48 meses    1770
Name: count, dtype: int64

In [14]:
tabela_permanencia = pd.crosstab(
    df_gold["Faixa_permanencia"],
    df_gold["Churn"]
)

tabela_permanencia = tabela_permanencia.rename(
    columns={"No": "Permaneceram", "Yes": "Cancelaram"}
)

tabela_permanencia["Total"] = tabela_permanencia.sum(axis=1)

tabela_permanencia["Taxa de cancelamento (%)"] = (
    tabela_permanencia["Cancelaram"] / tabela_permanencia["Total"] * 100
).round(2)

tabela_permanencia.index.name = "Tempo de permanência"
tabela_permanencia.columns.name = None

tabela_permanencia

,Permaneceram,Cancelaram,Total,Taxa de cancelamento (%)
Tempo de permanência,,,,
Até 12 meses,932,819,1751,46.77
13 a 24 meses,576,229,805,28.45
25 a 48 meses,1009,280,1289,21.72
Acima de 48 meses,1602,168,1770,9.49


### Resultado da análise de cancelamentos por faixa de permanência

Os clientes com até 12 meses de permanência apresentaram a maior taxa de cancelamento (46,77%), enquanto aqueles com mais de 48 meses registraram a menor taxa (9,49%).

Os resultados indicam que o cancelamento é proporcionalmente mais frequente entre clientes com menor tempo de permanência, sugerindo a importância de ações de retenção nos primeiros meses de relacionamento.

In [15]:
pasta_gold = Path("dados/gold")

pasta_gold.mkdir(parents=True, exist_ok=True)

tabela_contratos.to_csv(
    pasta_gold / "contratos.csv"
)

tabela_internet.to_csv(
    pasta_gold / "internet.csv"
)

tabela_permanencia.to_csv(
    pasta_gold / "permanencia.csv"
)

In [16]:
list(pasta_gold.iterdir())

[WindowsPath('dados/gold/.gitkeep'),
 WindowsPath('dados/gold/contratos.csv'),
 WindowsPath('dados/gold/internet.csv'),
 WindowsPath('dados/gold/permanencia.csv')]

### Finalização da camada Gold

A camada Gold foi organizada a partir dos dados tratados na Silver, gerando três tabelas com indicadores de cancelamento por tipo de contrato, serviço de internet e faixa de permanência.

Os arquivos `contratos.csv`, `internet.csv` e `permanencia.csv` foram salvos em `dados/gold/` e estão disponíveis para a construção de gráficos e análises exploratórias.